# Lesson 9A: Trust Region and PPO Methods - Theory

## Introduction

Policy gradient methods work but can be unstable with large updates. **Trust region methods** constrain updates to a "trust region" where the surrogate objective is reliable.

- **TRPO (Trust Region Policy Optimization)**: Enforce KL divergence constraint
- **PPO (Proximal Policy Optimization)**: Simpler clip-based constraint
- Industry standard: PPO is simpler and more widely used than TRPO

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Part 1: Importance Sampling and the Surrogate Objective

### Off-Policy Correction

Policy gradient under old policy π_old:
$$J(\theta) = \mathbb{E}_{s \sim \rho, a \sim \pi_{old}}\left[\frac{\pi_\theta(a|s)}{\pi_{old}(a|s)} A^{old}(s,a)\right]$$

This **importance-weighted surrogate objective** lets us reuse data from old policy π_old.

Key insight: if importance weights diverge far from 1, the estimate becomes unreliable → need trust region.

## Part 2: TRPO (Trust Region Policy Optimization)

### KL Divergence Constraint

Constrain the KL divergence between old and new policy:
$$\max_\theta \mathbb{E}[\frac{\pi_\theta(a|s)}{\pi_{old}(a|s)} A(s,a)]$$
$$\text{subject to } \mathbb{E}[\text{KL}(\pi_{old}||\pi_\theta)] \leq \delta$$

**Implementation**:
1. Compute conjugate gradient direction
2. Line search along that direction
3. Guarantee performance improvement

Advantage: **theoretical guarantees** on policy improvement
Disadvantage: Complex to implement (conjugate gradient, line search)

## Part 3: PPO (Proximal Policy Optimization)

### Clipped Surrogate Objective

Much simpler than TRPO. Instead of KL constraint, use a **clipped objective**:

$$L^{CLIP}(\theta) = \mathbb{E}[\min(r_t(\theta) A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) A_t)]$$

Where:
$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{old}(a_t|s_t)}$$

The `clip` function prevents the ratio from going too far from 1.

**Why it works**:
- If advantage > 0: want to increase π_θ, but clip at 1+ε
- If advantage < 0: want to decrease π_θ, but clip at 1-ε
- Natural constraint without explicit KL penalty

In [ ]:
# Visualize PPO clipping
def ppo_clip(ratio, advantage, epsilon=0.2):
    """PPO clipped objective."""
    clipped = np.clip(ratio, 1 - epsilon, 1 + epsilon)
    return np.minimum(ratio * advantage, clipped * advantage)

ratios = np.linspace(0.5, 1.5, 100)
advantage_pos = ppo_clip(ratios, 1.0, epsilon=0.2)
advantage_neg = ppo_clip(ratios, -1.0, epsilon=0.2)

plt.figure(figsize=(10, 5))
plt.plot(ratios, advantage_pos, label='Advantage > 0 (encourage action)', linewidth=2)
plt.plot(ratios, advantage_neg, label='Advantage < 0 (discourage action)', linewidth=2)
plt.axvline(x=1.0, color='k', linestyle='--', alpha=0.3, label='No change (ratio=1)')
plt.axvline(x=0.8, color='r', linestyle='--', alpha=0.3)
plt.axvline(x=1.2, color='r', linestyle='--', alpha=0.3)
plt.xlabel('Importance Ratio π(a|s) / π_old(a|s)')
plt.ylabel('Clipped Objective Value')
plt.title('PPO Clipping: Prevents Extreme Ratio Changes')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 4: PPO Algorithm

### Training Loop

1. Collect trajectories under current policy
2. Compute advantages (GAE or n-step)
3. For K epochs:
   - Sample minibatches
   - Compute clipped PPO loss
   - Compute value loss
   - Gradient step on both
4. Repeat

**Hyperparameters**:
- Clip ε: typically 0.1–0.2
- Learning rate: 3e-4 (adaptive fine-tuning needed)
- Epochs per batch: 3–10 (reuse data multiple times)
- Entropy coefficient: small (0.01) for exploration

In [ ]:
class PPOObjective:
    """PPO loss computation (pseudocode for theory)."""
    
    def __init__(self, clip_eps=0.2, entropy_coeff=0.01, value_coeff=0.5):
        self.clip_eps = clip_eps
        self.entropy_coeff = entropy_coeff
        self.value_coeff = value_coeff
    
    def ppo_loss(self, log_pi_new, log_pi_old, advantages):
        """Compute PPO clipped loss."""
        ratio = np.exp(log_pi_new - log_pi_old)
        clipped = np.clip(ratio, 1 - self.clip_eps, 1 + self.clip_eps)
        loss = -np.minimum(ratio * advantages, clipped * advantages).mean()
        return loss
    
    def value_loss(self, values, returns):
        """Value function MSE loss."""
        return ((values - returns) ** 2).mean()
    
    def entropy_bonus(self, probs):
        """Entropy regularization (exploration)."""
        entropy = -(probs * np.log(probs + 1e-8)).sum(axis=1).mean()
        return self.entropy_coeff * entropy
    
    def total_loss(self, log_pi_new, log_pi_old, advantages, values, returns, probs):
        """Combined objective: policy loss + value loss - entropy bonus."""
        actor_loss = self.ppo_loss(log_pi_new, log_pi_old, advantages)
        critic_loss = self.value_loss(values, returns)
        entropy = self.entropy_bonus(probs)
        return actor_loss + self.value_coeff * critic_loss - entropy

print("PPO objective: defined")

## Part 5: TRPO vs PPO Comparison

| Feature | TRPO | PPO |
|---------|------|-----|
| Constraint | KL divergence | Clipped ratio |
| Complexity | High (CG, line search) | Low (simple clip) |
| Sample efficiency | High | Medium-High |
| Implementation | Complex | Simple |
| Theory | Monotonic improvement | Approximate |
| Practice | Stable | Easier to tune |

**Verdict**: PPO is the industry standard for continuous control (robotics, games) because:
1. Simpler to implement
2. Empirically as good as TRPO
3. Easier hyperparameter tuning
4. More widely adopted → more resources

## Key Concepts

1. **Importance sampling**: Reuse old policy data via π_new / π_old
2. **Trust region**: Constrain policy change to stay near old policy
3. **TRPO**: KL constraint + conjugate gradient (theory-grounded)
4. **PPO**: Clipped ratio (practical alternative)
5. **PPO training**: Multiple epochs on same batch, value + policy loss

Next: Implement PPO in PyTorch on continuous control tasks (HalfCheetah, Hopper).